# Hands-On Machine Learning — Chapter 6
## Decision Trees


### Introduction

Decision Trees are powerful and interpretable models used for **classification** and **regression** tasks. They learn a hierarchy of if-then rules from data that recursively splits the feature space into regions with homogeneous labels or values.

Trees are intuitive, require little data preparation, and capture **nonlinear relationships** naturally. However, they are prone to overfitting unless regularized or averaged in ensembles like Random Forests.

<p align="left"><img src="../fig/figure6.1.png" width="45%"></p>

### Decision Tree Structure

A Decision Tree consists of:
- **Root node:** the top node representing the entire dataset.
- **Internal nodes:** test conditions on features.
- **Leaf nodes:** output predictions (class label or mean value).

Each internal node splits the dataset into two or more subsets based on a **decision rule** such as `feature ≤ threshold`.

During training, the algorithm recursively chooses the best feature and threshold that maximize the purity of resulting subsets.

<p align="left"><img src="../fig/figure6.2.png" width="45%"></p>

### How Trees Make Decisions

For classification, each node represents a subset of training samples. The algorithm chooses the split that best separates classes according to an **impurity measure** (e.g., Gini or entropy).

For regression, the goal is to minimize the variance of target values within each node.

Once trained, predictions are made by traversing the tree according to the feature values of a new instance until a leaf node is reached.

<p align="left"><img src="../fig/figure6.3.png" width="45%"></p>

### Training Algorithm — CART

Scikit-Learn implements the **CART (Classification And Regression Trees)** algorithm. It works as follows:
1. Start with the full training set.
2. For every feature and possible threshold, compute the split that minimizes the cost function.
3. Select the best split and partition the data.
4. Repeat recursively for each subset until stopping criteria are met.

For classification, the cost function is the weighted sum of child node impurities:

$$ J(k, t_k) = \frac{m_{left}}{m} G_{left} + \frac{m_{right}}{m} G_{right} $$

where $G$ can be **Gini impurity** or **entropy**.

<p align="left"><img src="../fig/figure6.4.png" width="45%"></p>

### Impurity Measures

1. **Gini Impurity** (default in Scikit-Learn):

$$ G = 1 - \sum_{k=1}^{K} p_k^2 $$
where $p_k$ is the fraction of samples of class $k$ in the node.

2. **Entropy:**

$$ H = -\sum_{k=1}^{K} p_k \log_2(p_k) $$

Gini and entropy usually produce similar trees, though Gini is slightly faster and tends to isolate the most frequent class first.

<p align="left"><img src="../fig/figure6.5.png" width="45%"></p>

### Visualizing Decision Boundaries

Decision Trees partition the feature space into rectangular regions. Visualizing these boundaries helps understand how the tree learns splits.

<p align="left"><img src="../fig/figure6.6.png" width="45%"></p>

In [ ]:
# Example: Decision Boundary Visualization for DecisionTreeClassifier
from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeClassifier
import numpy as np
import matplotlib.pyplot as plt

X, y = make_moons(n_samples=200, noise=0.25, random_state=42)
tree_clf = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_clf.fit(X, y)

x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))
Z = tree_clf.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.3)
plt.scatter(X[:, 0], X[:, 1], c=y, edgecolor='k', cmap=plt.cm.coolwarm)
plt.title('Decision Tree Decision Boundary (max_depth=4)')
plt.show()

### Regularization and Overfitting Control

Decision Trees can perfectly fit the training set (zero training error), leading to **overfitting**. Regularization restricts the tree’s growth:

- **max_depth:** limits the tree’s maximum depth.
- **min_samples_split:** minimum number of samples required to split an internal node.
- **min_samples_leaf:** minimum samples required to be a leaf node.
- **max_leaf_nodes:** limits the total number of leaves.
- **max_features:** number of features considered per split.

These hyperparameters help control complexity and improve generalization.

<p align="left"><img src="../fig/figure6.7.png" width="45%"></p>

### Regression Trees

Regression Trees predict continuous values by averaging targets in each leaf. Splits minimize the **Mean Squared Error (MSE)** of the target values:

$$ J = \frac{1}{m_{left}} \sum (y_i - \bar{y}_{left})^2 + \frac{1}{m_{right}} \sum (y_i - \bar{y}_{right})^2 $$

Unlike linear regression, Regression Trees can model discontinuous or nonlinear relationships.

<p align="left"><img src="../fig/figure6.8.png" width="45%"></p>

### Feature Importance

Decision Trees can quantify how much each feature contributes to reducing impurity, providing **feature importance scores**. The importance of a feature is the total reduction of impurity it brings, averaged over all trees in ensembles.


In [ ]:
# Feature Importance Example using Iris Dataset
from sklearn.datasets import load_iris

iris = load_iris()
tree_iris = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_iris.fit(iris.data, iris.target)

for name, score in zip(iris.feature_names, tree_iris.feature_importances_):
    print(f'{name}: {score:.4f}')

### Interpreting and Visualizing Trees

Decision Trees are highly interpretable: each node corresponds to a human-readable rule. They can be visualized using Scikit-Learn’s built-in tools:

- `plot_tree()` for quick visual inspection.
- `export_graphviz()` for detailed visualization with Graphviz.


In [ ]:
# Visualizing a Decision Tree using plot_tree
from sklearn.tree import plot_tree

plt.figure(figsize=(10, 6))
plot_tree(tree_iris, filled=True, feature_names=iris.feature_names, class_names=iris.target_names)
plt.show()

### Advantages and Limitations

**Advantages:**
- Easy to interpret and visualize.
- Handles numerical and categorical data.
- Requires little feature scaling or normalization.
- Can capture nonlinear patterns.

**Limitations:**
- High variance; small data changes can lead to different trees.
- Prone to overfitting without regularization.
- Decision boundaries are axis-aligned — not optimal for some data shapes.


### Random Forests — Ensemble of Trees

Random Forests combine multiple Decision Trees trained on random subsets of data and features. Predictions are averaged (regression) or voted (classification) among the ensemble.

This reduces variance and improves generalization. Random Forests are robust, efficient, and among the most popular algorithms in practice.


In [ ]:
# Random Forest Example
from sklearn.ensemble import RandomForestClassifier
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(iris.data, iris.target)
print('Random Forest Accuracy:', rf_clf.score(iris.data, iris.target))

### Summary and Insights

- Decision Trees recursively split data to create interpretable models.
- The CART algorithm chooses splits that minimize impurity.
- Regularization is essential to avoid overfitting.
- Trees can handle both classification and regression.
- Feature importance and visualizations make trees transparent.
- Ensembles like Random Forests overcome variance and boost performance.

